# Agent Evaluation

An agent can write a convincing answer while choosing the wrong tool, passing invented arguments, or reporting an action that never succeeded. Evaluation therefore needs two views: the quality of the final answer and the sequence of decisions that produced it.


## What Are We Evaluating?

Agent evaluation
: Measuring whether an agent completes a specified task, uses evidence and tools correctly, and produces an acceptable response under defined test conditions.

A useful evaluation starts with a concrete task and an observable success condition. “Is this good?” is too vague: a response can be fluent but incorrect, accurate but irrelevant, or correct only because a tool error happened not to matter.

| Dimension | Question | Observable evidence |
| --- | --- | --- |
| Task completion | Did the requested outcome happen? | Final state, returned artifact, or verified answer |
| Factuality | Are the claims supported? | Claims checked against reliable evidence |
| Relevance and usefulness | Does the answer address the request? | Task-specific rubric |
| Tool use | Were the tool and arguments appropriate? | Tool-call trace and results |
| Response requirements | Did the response follow the required format and tone? | Schema checks and rubric judgments |
| System performance | What did the run cost? | Latency, calls, failures, and resource usage |

:::{admonition} A Successful Call Is Not a Successful Task
:class: important

A tool may execute without error and still do the wrong thing. Conversely, a correct agent may report that a requested action cannot proceed because required information is missing. Grade the behavior against the task and available information, rather than rewarding every apparent completion.
:::


## Human Ratings and Agreement

Human evaluation is useful for open-ended qualities such as helpfulness, but different raters may interpret the same criterion differently. “Suggest a useful birthday gift” leaves room for disagreement about specificity, suitability, and whether the agent should ask a follow-up question.

A **rubric** makes these expectations explicit. Raters can compare disagreements, refine ambiguous wording, and then evaluate fresh cases. Agreement measures help identify unclear criteria; they do not prove that a shared judgment is correct.

Inter-rater agreement
: The extent to which different raters assign the same labels to the same items under a shared rating procedure.

### Agreement Beyond Chance

Let $p_o$ be the observed fraction of matching labels. If rater A assigns “pass” with probability $p_A$ and rater B does so with probability $p_B$, independent labels with those marginal probabilities would agree at rate

$$
p_e=p_Ap_B+(1-p_A)(1-p_B).
$$

For two raters and categorical labels, Cohen's kappa adjusts observed agreement relative to this baseline:

$$
\kappa=\frac{p_o-p_e}{1-p_e}.
$$

Here $\kappa=1$ means perfect agreement, $\kappa=0$ means agreement equal to the marginal-based chance baseline, and negative values mean less agreement than that baseline. The expression is undefined if $p_e=1$.

:::{admonition} Worked Agreement Example
:class: tip ds-example-card

In an illustrative set of 100 answers, the raters produce these labels:

| | B: pass | B: fail | Total |
| --- | ---: | ---: | ---: |
| A: pass | 50 | 10 | 60 |
| A: fail | 10 | 30 | 40 |
| Total | 60 | 40 | 100 |

Observed agreement is $p_o=(50+30)/100=0.80$. Both raters pass 60% of answers, so $p_e=0.6^2+0.4^2=0.52$. Therefore

$$
\kappa=\frac{0.80-0.52}{1-0.52}\approx0.583.
$$

The raw 80% agreement has to be interpreted alongside the label frequencies. For example, two independent raters who each pass 90% of answers already have expected agreement $0.9^2+0.1^2=0.82$.
:::

Other agreement measures accommodate different settings, such as multiple raters or missing ratings. Choose the measure for the annotation design; do not apply one universal cutoff to every task.


## Reference-Based Metrics

Instead of asking people to rate every new output, one can prepare reference answers and compare model outputs against them repeatedly. This is efficient when the expected answer is well specified. Natural language complicates the comparison because different wording can express the same meaning.

| Method | What it measures | Main limitation |
| --- | --- | --- |
| Exact match | Whether a normalized answer matches the reference | Rejects valid alternative wording unless explicitly allowed |
| BLEU | Modified n-gram precision with a brevity penalty | Overlap does not establish factuality or task completion |
| METEOR | Unigram alignment, precision/recall balance, and ordering | Depends on references, matching rules, and parameter choices |
| Executable checks | Whether an artifact passes a test | Tests may miss important cases |

### How METEOR Balances Matches and Order

Let $m$ be the number of aligned unigrams, $L_c$ the candidate length, and $L_r$ the reference length. Then $P=m/L_c$ and $R=m/L_r$. A parameterized form is

$$
F_{\alpha}=\frac{PR}{\alpha P+(1-\alpha)R},\qquad
\mathrm{Penalty}=\gamma\left(\frac{c}{m}\right)^\beta,
$$

$$
\mathrm{METEOR}=(1-\mathrm{Penalty})F_{\alpha}.
$$

Here $c$ counts contiguous aligned chunks; $\alpha,\beta,\gamma$ control the balance and penalty. Fewer chunks for the same number of matched words indicate better preservation of order. With no matches, the score is zero rather than evaluating $c/m$.

:::{admonition} Match the Metric to the Claim
:class: note

A high text-overlap score is evidence of similarity to a reference. It is not evidence that a booking happened, a tool returned the cited fact, or an answer is useful. Agent tasks often require state checks and trace inspection alongside text scoring.
:::


## LLM as a Judge

LLM as a judge
: A model used to evaluate another response against explicit criteria, producing a judgment and a short explanation tied to the evidence.

The judge receives the original task, the candidate response, and the rubric. For grounded tasks, it also needs relevant evidence or tool results. Otherwise it may reward a plausible unsupported answer.

```{figure} figures/evaluation-judge.svg
:alt: Task and response flow into a rubric-based judge, then a validated result, then human calibration.
The judge is part of the measurement system and must itself be checked.
```

::::{div}
:class: reading-comparison

:::{div}
### Single-Response Grading

Score one answer against a criterion, such as whether every stated completion is supported by a successful tool result. A pass/fail decision is useful when the criterion has a clear boundary.
:::

:::{div}
### Pairwise Comparison

Compare two answers under the same rubric. Allow a tie or inconclusive outcome where appropriate. Relative preference alone does not establish that either answer is acceptable.
:::
::::

### Make the Judgment Inspectable

A compact judgment can contain an evidence-based explanation followed by the decision:

```json
{
  "explanation": "The answer claims completion, but the tool returned a permission error.",
  "decision": "fail"
}
```

Prompting for JSON alone does not ensure valid JSON. Schema-constrained generation can restrict the output structure; the application still needs to handle refusal, truncation, and failed calls. A valid schema does not make the judgment true.

The explanation should identify the decisive evidence. It need not expose a long internal reasoning trace. Whether requesting an explanation improves judgments should be tested on the actual evaluation set.

### Bias and Calibration

| Failure mode | What it looks like | Practical diagnostic |
| --- | --- | --- |
| Position bias | Preferring the first answer regardless of content | Repeat with answer order swapped and map results back to the original identities |
| Verbosity bias | Rewarding length instead of usefulness | Compare concise and verbose answers with equivalent substance |
| Self-preference | Favoring outputs similar to the judge's own style | Compare against human labels and an independent judge |
| Rubric mismatch | Applying a different standard from the intended one | Inspect disagreements on a human-labeled sample |

If swapped comparisons disagree, mark the result unstable or adjudicate it. Two opposing votes do not create a majority. A different or more capable judge may help, but does not remove the need for calibration.

:::{admonition} The Judge Score Is a Proxy
:class: warning

Repeatedly optimizing against one judge can improve its score without improving the user experience. Keep separate human-reviewed cases for checking the judge, and distinguish false passes from false failures. Record the rubric, judge version, and generation settings. Low temperature can reduce variability, but does not guarantee identical runs.
:::


## Factuality at the Claim Level

A paragraph may mix correct, incorrect, and unresolved statements. Decomposing it into atomic claims makes partial errors visible and avoids treating every paragraph as an indivisible pass/fail item.

Atomic claim
: A statement small enough to be checked as a single factual assertion against relevant evidence.

```{figure} figures/evaluation-facts.svg
:alt: Split an answer into claims, retrieve evidence, label each claim, then aggregate supported claims.
Keep the claim-to-evidence connection so a score can be traced back to its basis.
```

For claim labels $s_i\in\{0,1\}$ and nonnegative importance weights $w_i$, a normalized supported-claim score is

$$
S=\frac{\sum_{i=1}^{n}w_i s_i}{\sum_{i=1}^{n}w_i},\qquad \sum_i w_i>0.
$$

A weighted sum alone is comparable across answers only if its normalization is controlled. Define weights before judging the candidates. Also track unresolved claims: failure to find evidence is not automatically proof that a statement is false.

:::{admonition} Worked Factuality Example
:class: tip ds-example-card

Suppose a fictional agent reports: “The reservation is confirmed for Tuesday at 7 pm for four guests.” The tool record says Tuesday at 7 pm, **two** guests, with status **pending**.

| Claim | Evidence label | Weight | Supported contribution |
| --- | --- | ---: | ---: |
| Tuesday | Supported | 1 | 1 |
| 7 pm | Supported | 1 | 1 |
| Four guests | Contradicted | 2 | 0 |
| Confirmed | Contradicted | 2 | 0 |

The supported-claim score is $S=2/6=1/3$. More importantly, the action-success criterion fails: the reservation is not confirmed. Correct details must not compensate for a critical false completion claim.
:::

Claim coverage matters too. A response that omits the difficult parts may score highly on the few claims it makes while failing to answer the question. Evaluate completeness separately.


## Evaluating the Agent Trace

An agent repeats a loop of choosing an action, executing it, interpreting the observation, and deciding what to do next. A final-answer score tells us whether the run looks successful; the trace helps explain why it succeeded or failed.

Execution trace
: The recorded sequence of available tools, selected calls, arguments, returned results, and agent responses during a run.

```{figure} figures/evaluation-trace.svg
:alt: Check tool selection, execution, interpretation, and the final task outcome, in that order.
Find the earliest meaningful divergence before changing the whole system.
```

### Seven Common Failure Modes

| Stage | Failure | What to inspect | Possible repair |
| --- | --- | --- | --- |
| Selection | No tool used when one is needed | Was the required tool exposed to the model? | Improve router recall or clarify when the tool is needed |
| Selection | Invented tool name | Selected name versus available schema | Clarify names, descriptions, and allowed calls |
| Selection | Wrong existing tool | Overlapping responsibilities and tool descriptions | Separate tool scope and routing criteria |
| Arguments | Right tool, wrong inputs | Required context, units, identifiers, and argument schema | Supply missing context or ask for it; validate inputs |
| Execution | Tool returns an incorrect result | Backend behavior with known-good inputs | Fix the implementation or external dependency |
| Execution | Tool provides no meaningful result | Whether success, no results, and errors are distinguishable | Return an explicit status and relevant data |
| Synthesis | Answer contradicts the tool result | Result size, structure, and grounding | Trim irrelevant output and expose the decisive fields |

A router is a component that selects which tools to make available. A recall-oriented router tries to retain the tools needed for the request while reducing context size. If the correct tool never reaches the model, rewriting the final-answer prompt addresses the wrong stage.

:::{admonition} Missing Context Is Not an Argument Value
:class: note

For “find a café near me,” coordinates $(0,0)$ are not a substitute for an unknown location. The agent should obtain permitted location information or ask the user. Include missing-context cases in evaluation so invented defaults cannot pass unnoticed.
:::

### Distinguish No Results from Failure

An explicit response is easier to interpret than silence or an ambiguous empty object:

```json
{
  "status": "ok",
  "results": [],
  "message": "No matching cafés were found within the requested radius."
}
```

This means a search completed with no matches. A permission failure should have a different status. For actions, return confirmed state or a verifiable identifier; do not treat the absence of an error message as proof of success.


## From One Failure to a Regression Test

A useful test case fixes the task, initial context, available tools, and expected behavior. Controlled tool fixtures isolate decision errors; end-to-end tests additionally check whether the real backend and integrations work.

:::{admonition} Worked Trace Diagnosis
:class: tip ds-example-card

**Task:** Find an open café within one kilometre of the supplied location.

**Observed trace:** The agent calls the correct search tool with the correct location and radius. The tool returns two cafés, both marked closed. The answer recommends one as open.

**Diagnosis:** Selection and argument construction passed. The failure is in interpreting the result and making a supported recommendation.

**Expected behavior:** Report that no open café was found within the requested radius. A larger-radius search may be offered or performed only when the task permits it.

**Regression case:** Replay the same closed-café results after a change. Check that the agent does not describe either café as open. Add a companion case with one open café so a model that always says “none found” cannot pass the whole test set.
:::

Track results by failure category rather than only one average. Keep task completion, groundedness, latency, and cost visible separately. A cheaper run that skips a required action is not an equivalent improvement.


## Benchmarks and What They Establish

Benchmark
: A defined set of tasks, execution conditions, and scoring rules used to compare systems under a shared protocol.

Knowledge questions, reasoning problems, code tests, and interactive tasks measure different capabilities. A multiple-choice knowledge benchmark such as MMLU can use mechanically checked answers, reducing dependence on another model's judgment. Its score does not directly measure reliable tool use in a particular application.

| Test setup | What changes | What must stay clear |
| --- | --- | --- |
| Zero-shot | No worked demonstrations in the prompt | Instructions and allowed context |
| Few-shot | Demonstrations are supplied in the prompt | Which examples and how many |
| Fine-tuned | Model parameters have been updated | Training data must remain separate from evaluation cases |

Accuracy is the fraction of correct predictions. Recall is $\mathrm{TP}/(\mathrm{TP}+\mathrm{FN})$, not the raw count of true positives. Perplexity measures predictive fit to token sequences; it is not a direct measure of agent task completion. Benchmark scores need not all use a 0–100 scale.

:::{admonition} Read the Conditions Behind the Score
:class: note

A benchmark can miss rare failures, become saturated, or overlap with training data. Compare systems under compatible prompts, tool access, budgets, and scoring rules. For an application decision, complement broad benchmark results with representative local tasks and repeated runs where outcomes vary.
:::
